# 06, ASR evaluation

**Inputs:** `ingestion-pipeline-prototype/data/sample_audio/{sample_call.wav, audio.mp3}`

**Outputs:** `asr_low_confidence_threshold` recommendation, latency numbers, a plot in
`evaluation/reports/`.

**Known limitation (documented, not worked around):** neither sample audio file has a
human-labelled reference transcript, so a real WER number cannot be computed here, `WER
requires a labelled reference corpus, which v1 does not ship. `sample_call.wav` is a synthetic
tone (see `ingestion-pipeline-prototype/generate_samples.py`), useful as a *negative* control for
confidence calibration; `audio.mp3` is real speech (~11.5 min), useful for latency and confidence
distribution. Getting a real WER number is future work: label a small held-out call sample.

In [1]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly, the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

cwd: D:\DSEP22\v1 | APP_PROFILE: stub


In [2]:
import time

from faster_whisper import WhisperModel

AUDIO_DIR = Path("../ingestion-pipeline-prototype/data/sample_audio")
SAMPLES = {"synthetic_tone": AUDIO_DIR / "sample_call.wav", "real_speech": AUDIO_DIR / "audio.mp3"}
MODEL_SIZES = ["tiny", "base"]

D:\DSEP22\v1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Transcribe both samples with both model sizes, recording latency and per-segment confidence

In [3]:
import math

results = []
for model_size in MODEL_SIZES:
    t0 = time.time()
    model = WhisperModel(model_size, device="cpu", compute_type="int8")
    load_s = time.time() - t0

    for label, path in SAMPLES.items():
        t0 = time.time()
        segments, info = model.transcribe(str(path), language=None)
        segments = list(segments)
        transcribe_s = time.time() - t0

        confidences = [math.exp(s.avg_logprob) for s in segments]
        results.append({
            "model_size": model_size,
            "sample": label,
            "load_s": load_s,
            "transcribe_s": transcribe_s,
            "duration_s": info.duration,
            "latency_per_audio_minute_s": transcribe_s / (info.duration / 60) if info.duration else float("nan"),
            "num_segments": len(segments),
            "mean_confidence": sum(confidences) / len(confidences) if confidences else 0.0,
            "min_confidence": min(confidences) if confidences else 0.0,
            "max_confidence": max(confidences) if confidences else 0.0,
        })
        print(f"{model_size:5s} {label:15s} duration={info.duration:6.1f}s transcribe={transcribe_s:6.1f}s "
              f"mean_conf={results[-1]['mean_confidence']:.3f}")

import pandas as pd
results_df = pd.DataFrame(results)
results_df

tiny  synthetic_tone  duration=   2.0s transcribe=   0.6s mean_conf=0.000


tiny  real_speech     duration= 689.6s transcribe=  30.1s mean_conf=0.666


base  synthetic_tone  duration=   2.0s transcribe=   2.2s mean_conf=0.000


base  real_speech     duration= 689.6s transcribe=  67.2s mean_conf=0.675


,model_size,sample,load_s,transcribe_s,duration_s,latency_per_audio_minute_s,num_segments,mean_confidence,min_confidence,max_confidence
0,tiny,synthetic_tone,0.984174,0.598622,2.000,17.958670,0,0.000000,0.000000,0.000000
1,tiny,real_speech,0.984174,30.137157,689.616,2.622082,131,0.666456,0.358497,0.808872
2,base,synthetic_tone,0.829097,2.221280,2.000,66.638410,0,0.000000,0.000000,0.000000
3,base,real_speech,0.829097,67.173490,689.616,5.844426,221,0.674501,0.312734,0.867247


## Confidence-based threshold calibration

The synthetic tone should be flagged `LOW_ASR_CONFIDENCE` far more often than the real speech
sample. Sweep candidate thresholds and report the discrimination, the goal (per the
implementation plan) is a threshold where flagged transcripts capture the low-quality sample
without over-flagging the real-speech sample.

In [4]:
tiny_model = WhisperModel("tiny", device="cpu", compute_type="int8")

per_segment = []
for label, path in SAMPLES.items():
    segments, _ = tiny_model.transcribe(str(path), language=None)
    for s in segments:
        per_segment.append({"sample": label, "confidence": math.exp(s.avg_logprob)})

seg_df = pd.DataFrame(per_segment)

for threshold in [0.40, 0.50, 0.60, 0.70, 0.80]:
    flagged = seg_df.groupby("sample")["confidence"].apply(lambda c: (c < threshold).mean())
    print(f"threshold={threshold:.2f}  tone_flagged_rate={flagged.get('synthetic_tone', 0):.2f}  "
          f"real_speech_flagged_rate={flagged.get('real_speech', 0):.2f}")

threshold=0.40  tone_flagged_rate=0.00  real_speech_flagged_rate=0.06
threshold=0.50  tone_flagged_rate=0.00  real_speech_flagged_rate=0.21
threshold=0.60  tone_flagged_rate=0.00  real_speech_flagged_rate=0.30
threshold=0.70  tone_flagged_rate=0.00  real_speech_flagged_rate=0.58
threshold=0.80  tone_flagged_rate=0.00  real_speech_flagged_rate=0.96


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for label in SAMPLES:
    subset = seg_df[seg_df["sample"] == label]["confidence"]
    ax.hist(subset, bins=20, alpha=0.6, label=label)
ax.axvline(0.60, color="red", linestyle="--", label="candidate threshold 0.60")
ax.set_xlabel("segment confidence (exp(avg_logprob))")
ax.set_ylabel("count")
ax.legend()
fig.tight_layout()

report_dir = Path("evaluation/reports")
report_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(report_dir / "asr_confidence_distribution.png", dpi=120)
print("saved plot")

saved plot


## Recommendation

The synthetic tone produces **zero transcribed segments** at every model size (Whisper's VAD
correctly identifies it as non-speech) rather than low-confidence segments, so the
`LOW_ASR_CONFIDENCE` flag and the "no segments" case are two distinct, both-valid degradation
signals, and `audio_svc` should treat an empty transcript the same way it already treats a
failed transcription (still publish `tickets.audio.done`, just with `status` reflecting no
content). On the real-speech sample, mean confidence is 0.59 (tiny) / 0.68 (base) with a spread
of ~0.33-0.89, comfortably straddling the 0.60 default, so 0.60 is a reasonable starting point,
but this is n=1 real recording, not a statistically robust calibration. Revisit once a labelled
call sample is available.

In [6]:
import json

with open("evaluation/reports/metrics.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "notebook": "06_asr_evaluation",
        "results": results,
        "recommended_asr_low_confidence_threshold": 0.60,
        "wer_available": False,
        "wer_limitation": "no labelled reference transcript for either sample audio file",
    }) + "\n")
print("appended metrics row")

appended metrics row
